In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.3562335669994354
epoch:  1, loss: 0.29379773139953613
epoch:  2, loss: 0.2543405294418335
epoch:  3, loss: 0.14245140552520752
epoch:  4, loss: 0.05468125268816948
epoch:  5, loss: 0.026005461812019348
epoch:  6, loss: 0.011677891947329044
epoch:  7, loss: 0.007819590158760548
epoch:  8, loss: 0.0063766492530703545
epoch:  9, loss: 0.005552827846258879
epoch:  10, loss: 0.004798966459929943
epoch:  11, loss: 0.0044546122662723064
epoch:  12, loss: 0.003918766044080257
epoch:  13, loss: 0.003644638927653432
epoch:  14, loss: 0.0033739926293492317
epoch:  15, loss: 0.0030954363755881786
epoch:  16, loss: 0.002892036223784089
epoch:  17, loss: 0.002673547016456723
epoch:  18, loss: 0.002502461662515998
epoch:  19, loss: 0.0023489377927035093
epoch:  20, loss: 0.00221618777140975
epoch:  21, loss: 0.0021043969318270683
epoch:  22, loss: 0.0019936079625040293
epoch:  23, loss: 0.0018962851027026772
epoch:  24, loss: 0.001839929260313511
epoch:  25, loss: 0.001770005212165

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9762844572683337
Test metrics:  R2 = 0.860165032289772


In [7]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()

opt = torch_numopt.NewtonLS(model=model, batch_size=100)

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.24435874819755554
epoch:  1, loss: 0.21087665855884552
epoch:  2, loss: 0.13291534781455994
epoch:  3, loss: 0.11212607473134995
epoch:  4, loss: 0.09266949445009232
epoch:  5, loss: 0.07767166197299957
epoch:  6, loss: 0.06619594246149063
epoch:  7, loss: 0.02418077178299427
epoch:  8, loss: 0.009765769354999065
epoch:  9, loss: 0.005661172792315483
epoch:  10, loss: 0.004336806479841471
epoch:  11, loss: 0.0036911063361912966
epoch:  12, loss: 0.0032902411185204983
epoch:  13, loss: 0.0029830089770257473
epoch:  14, loss: 0.0027736257761716843
epoch:  15, loss: 0.0026049467269331217
epoch:  16, loss: 0.0024914233945310116
epoch:  17, loss: 0.002394154667854309
epoch:  18, loss: 0.0023036685306578875
epoch:  19, loss: 0.0022380512673407793
epoch:  20, loss: 0.002188681624829769
epoch:  21, loss: 0.002165765268728137
epoch:  22, loss: 0.002110476139932871
epoch:  23, loss: 0.002072589937597513
epoch:  24, loss: 0.0020352108404040337
epoch:  25, loss: 0.00200382596813

In [8]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9582680716666231
Test metrics:  R2 = 0.8343716492910322
